# EEG Epilepsy vs. Healthy Classification
**Dataset:** Zenodo 3684992 — *Single electrode EEG data of healthy and epileptic patients*  
Panwar et al., IEEE Trans. Neural Syst. Rehabil. Eng. 27, 1106–1116 (2019)

> Run cells sequentially. Only one extra install is needed (`antropy`). All other packages are pre-installed on Google Colab.

In [1]:
# ==============================================================================
# EEG Epilepsy vs. Healthy Classification — Google Colab Notebook
# Dataset: Zenodo 3684992 — "Single electrode EEG data of healthy and epileptic patients"
#          Panwar et al., IEEE Trans. Neural Syst. Rehabil. Eng. 27, 1106–1116 (2019)
#
# This notebook:
#   1. Downloads all TrainE* and TrainH* files directly from Zenodo
#   2. Extracts a rich feature set (time-domain, frequency-domain, nonlinear)
#   3. Performs comparative statistical analysis (E vs H)
#   4. Trains and evaluates multiple classifiers; selects the best
#   5. Provides a ready-to-use inference function for new user-supplied data
# ==============================================================================

## Install / import dependencies

In [2]:
# ─── Cell 1: Install / import dependencies ─────────────────────────────────────
# (All packages are pre-installed on Colab; antropy is the only extra one)
!pip install antropy -q

import os, re, io, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.signal import welch, periodogram
from scipy.stats import kurtosis, skew
import antropy as ant                     # approximate & sample entropy, Hurst
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_auc_score,
                              roc_curve, accuracy_score)
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings("ignore")

print("All imports OK ✓")

All imports OK ✓


## Download data from Zenodo

In [ ]:
# ─── Cell 2: Download data from Zenodo ──────────────────────────────────────────
ZENODO_BASE = "https://zenodo.org/records/3684992/files/"
DATA_DIR    = "eeg_data"
os.makedirs(DATA_DIR, exist_ok=True)

FS = 173.61          # sampling frequency (Hz) — Bonn/Panwar dataset
N_SAMPLES = 4097     # fixed length per segment

def build_file_list():
    """Return list of (url, local_path, label) for all TrainE and TrainH files."""
    entries = []
    for group, label in [("TrainE", 1), ("TrainH", 0)]:
        for subject in range(1, 6):          # subjects 1–5 per group
            for seg in range(1, 41):         # 40 segments per subject
                fname = f"{group}{subject}_{seg}.txt"
                url   = ZENODO_BASE + fname + "?download=1"
                path  = os.path.join(DATA_DIR, fname)
                entries.append((url, path, label, group+str(subject), fname))
    return entries

def download_file(url, path):
    if not os.path.exists(path):
        try:
            urllib.request.urlretrieve(url, path)
            return True
        except Exception as e:
            print(f"  ✗ Failed: {os.path.basename(path)} — {e}")
            return False
    return True

print("Building file list …")
file_list = build_file_list()
print(f"  Total files to download: {len(file_list)}")
print("Downloading (this may take 1–3 minutes on Colab) …")

ok = 0
for url, path, label, subj, fname in file_list:
    if download_file(url, path):
        ok += 1

print(f"\nDownloaded/available: {ok}/{len(file_list)} files")

Building file list …
  Total files to download: 400


## Load all time series

In [ ]:
# ─── Cell 3: Load all time series ───────────────────────────────────────────────
def load_txt_series(path):
    """Load a single-column EEG text file into a numpy array."""
    vals = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                vals.append(float(line))
            except ValueError:
                pass
    return np.asarray(vals, dtype=np.float64)

records = []   # list of dicts: {x, label, subject, filename}
for url, path, label, subj, fname in file_list:
    if os.path.exists(path):
        x = load_txt_series(path)
        if len(x) >= 1000:           # sanity check
            records.append({"x": x, "label": label,
                             "subject": subj, "filename": fname})

labels_str = {0: "Healthy", 1: "Epileptic"}
n_E = sum(1 for r in records if r["label"] == 1)
n_H = sum(1 for r in records if r["label"] == 0)
print(f"Loaded segments: Epileptic={n_E}, Healthy={n_H}, Total={len(records)}")

## Visualise representative segments

In [ ]:
# ─── Cell 4: Visualise representative segments ──────────────────────────────────
t = np.arange(N_SAMPLES) / FS          # time axis in seconds

fig, axes = plt.subplots(2, 2, figsize=(14, 6))
for i, (grp_label, grp_name) in enumerate([(1, "Epileptic (TrainE)"),
                                            (0, "Healthy (TrainH)")]):
    samples = [r for r in records if r["label"] == grp_label][:2]
    for j, rec in enumerate(samples):
        ax = axes[i][j]
        t_seg = np.arange(len(rec["x"])) / FS
        ax.plot(t_seg, rec["x"], lw=0.6, color="crimson" if grp_label else "steelblue")
        ax.set_title(f"{grp_name} — {rec['filename']}", fontsize=9)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Amplitude (µV)")
        ax.set_xlim(0, t_seg[-1])

plt.suptitle("Representative EEG segments", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("eeg_representative.png", dpi=150)
plt.show()

## Feature extraction

In [ ]:
# ─── Cell 5: Feature extraction ─────────────────────────────────────────────────
def hjorth(x):
    """Hjorth activity, mobility, complexity via antropy."""
    activity = float(np.var(x, ddof=1))
    mobility, complexity = ant.hjorth_params(x)
    return activity, float(mobility), float(complexity)

def band_power(f, Pxx, fmin, fmax):
    """Integrate PSD in a frequency band using trapezoidal rule."""
    mask = (f >= fmin) & (f <= fmax)
    return np.trapz(Pxx[mask], f[mask]) if mask.sum() > 1 else 0.0

def psd_slope(f, Pxx, fmin=2.0, fmax=40.0):
    """Log-log spectral slope in [fmin, fmax] Hz."""
    mask = (f >= fmin) & (f <= fmax) & (Pxx > 0)
    if mask.sum() < 4:
        return np.nan
    slope, _, r, _, _ = stats.linregress(np.log10(f[mask]), np.log10(Pxx[mask]))
    return slope

def extract_features(x, fs=FS):
    feats = {}

    # ── Time-domain ──────────────────────────────────────────────────────────
    feats["mean"]         = x.mean()
    feats["std"]          = x.std(ddof=1)
    feats["variance"]     = x.var(ddof=1)
    feats["skewness"]     = float(skew(x))
    feats["kurtosis"]     = float(kurtosis(x, fisher=True))
    feats["rms"]          = np.sqrt(np.mean(x**2))
    feats["peak_to_peak"] = x.max() - x.min()
    feats["iqr"]          = np.percentile(x, 75) - np.percentile(x, 25)
    feats["mad"]          = np.median(np.abs(x - np.median(x)))
    feats["zero_crossings"] = ((x[:-1] * x[1:]) < 0).sum()

    # Lag-1 autocorrelation and increment stats
    feats["acf1"]         = np.corrcoef(x[:-1], x[1:])[0, 1]
    dx = np.diff(x)
    feats["diff_std"]     = dx.std(ddof=1)
    feats["diff_kurtosis"] = float(kurtosis(dx, fisher=True))

    # Hjorth parameters
    act, mob, comp = hjorth(x)
    feats["hjorth_activity"]   = act
    feats["hjorth_mobility"]   = mob
    feats["hjorth_complexity"] = comp

    # ── Frequency-domain (Welch PSD) ─────────────────────────────────────────
    nperseg = min(1024, len(x))
    f, Pxx  = welch(x, fs=fs, nperseg=nperseg, noverlap=nperseg//2)
    total_power = np.trapz(Pxx, f) + 1e-12

    # EEG frequency bands
    bands = {
        "delta": (0.5, 4),
        "theta": (4, 8),
        "alpha": (8, 13),
        "beta":  (13, 30),
        "gamma": (30, 60),
    }
    for band, (lo, hi) in bands.items():
        bp = band_power(f, Pxx, lo, hi)
        feats[f"bp_{band}"]         = bp
        feats[f"bp_{band}_rel"]     = bp / total_power

    feats["spectral_slope"]     = psd_slope(f, Pxx, 2.0, 40.0)
    feats["dominant_frequency"] = f[np.argmax(Pxx)]
    feats["spectral_centroid"]  = np.sum(f * Pxx) / (np.sum(Pxx) + 1e-12)
    feats["spectral_entropy"]   = ant.spectral_entropy(x, sf=fs, method="welch", normalize=True)
    feats["total_power"]        = total_power

    # ── Nonlinear / entropy features ─────────────────────────────────────────
    feats["approximate_entropy"] = ant.app_entropy(x)
    feats["sample_entropy"]      = ant.sample_entropy(x)
    feats["perm_entropy"]        = ant.perm_entropy(x, normalize=True)
    feats["svd_entropy"]         = ant.svd_entropy(x, normalize=True)
    feats["hurst_exp"]           = ant.higuchi_fd(x)  # fractal dim (replaces Hurst)
    feats["detrended_fa"]        = ant.detrended_fluctuation(x)

    return feats

print("Extracting features … (may take 2–5 minutes)")
feature_rows = []
for rec in records:
    try:
        f = extract_features(rec["x"])
        f["label"]    = rec["label"]
        f["subject"]  = rec["subject"]
        f["filename"] = rec["filename"]
        feature_rows.append(f)
    except Exception as e:
        print(f"  Skipped {rec['filename']}: {e}")

df = pd.DataFrame(feature_rows)
df.to_csv("eeg_features.csv", index=False)
print(f"Feature matrix shape: {df.shape}")
print(df.groupby("label")[["acf1","diff_std","spectral_slope",
                            "approximate_entropy","hurst_exp"]].mean().rename(
    index={0: "Healthy", 1: "Epileptic"}))

## Statistical comparison E vs H

In [ ]:
# ─── Cell 6: Statistical comparison E vs H ──────────────────────────────────────
feature_cols = [c for c in df.columns
                if c not in ("label", "subject", "filename")]

E_df = df[df["label"] == 1][feature_cols]
H_df = df[df["label"] == 0][feature_cols]

stats_rows = []
for col in feature_cols:
    e_vals = E_df[col].dropna().values
    h_vals = H_df[col].dropna().values
    u_stat, p_mw = stats.mannwhitneyu(e_vals, h_vals, alternative="two-sided")
    cohens_d = (e_vals.mean() - h_vals.mean()) / np.sqrt(
        ((len(e_vals)-1)*e_vals.std(ddof=1)**2 + (len(h_vals)-1)*h_vals.std(ddof=1)**2)
        / (len(e_vals) + len(h_vals) - 2)
    )
    stats_rows.append({
        "feature": col,
        "mean_E": e_vals.mean(), "std_E": e_vals.std(ddof=1),
        "mean_H": h_vals.mean(), "std_H": h_vals.std(ddof=1),
        "MWU_pvalue": p_mw,
        "cohens_d": cohens_d,
        "abs_d": abs(cohens_d),
    })

stats_df = pd.DataFrame(stats_rows).sort_values("abs_d", ascending=False)
stats_df.to_csv("eeg_feature_statistics.csv", index=False)

print("\nTop 15 most discriminating features (by |Cohen d|):")
print(stats_df[["feature","mean_E","mean_H","MWU_pvalue","cohens_d"]].head(15).to_string(index=False))

## Visualise top features

In [ ]:
# ─── Cell 7: Visualise top features ─────────────────────────────────────────────
top_feats = stats_df.head(12)["feature"].tolist()
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.ravel()
palette = {0: "steelblue", 1: "crimson"}
for i, feat in enumerate(top_feats):
    ax = axes[i]
    for lbl, clr in palette.items():
        vals = df[df["label"] == lbl][feat].dropna()
        ax.hist(vals, bins=25, alpha=0.6, color=clr,
                label=labels_str[lbl], density=True)
    ax.set_title(feat.replace("_", " "), fontsize=8)
    ax.set_yticks([])
    if i == 0:
        ax.legend(fontsize=7)

plt.suptitle("Distribution of top discriminating features (E vs H)", fontweight="bold")
plt.tight_layout()
plt.savefig("feature_distributions.png", dpi=150)
plt.show()

# Heatmap of mean feature values (normalised)
top20 = stats_df.head(20)["feature"].tolist()
grp_means = df.groupby("label")[top20].mean().rename(index=labels_str)
normed = (grp_means - grp_means.min()) / (grp_means.max() - grp_means.min() + 1e-12)
fig, ax = plt.subplots(figsize=(14, 3))
sns.heatmap(normed, ax=ax, cmap="RdBu_r", annot=True, fmt=".2f", linewidths=0.5)
ax.set_title("Normalised mean feature values — Epileptic vs Healthy", fontweight="bold")
plt.tight_layout()
plt.savefig("feature_heatmap.png", dpi=150)
plt.show()

## Correlation heat-map within top features

In [ ]:
# ─── Cell 8: Correlation heat-map within top features ──────────────────────────
corr = df[top20].corr()
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=ax, cmap="coolwarm", center=0,
            linewidths=0.3, annot=False)
ax.set_title("Feature correlation matrix (top 20 features)", fontweight="bold")
plt.tight_layout()
plt.savefig("feature_correlation.png", dpi=150)
plt.show()

## PSD comparison

In [ ]:
# ─── Cell 9: PSD comparison ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for group_label, group_name, color, ax in [
    (1, "Epileptic", "crimson", axes[0]),
    (0, "Healthy",   "steelblue", axes[1]),
]:
    segs = [r for r in records if r["label"] == group_label][:20]
    psds = []
    for rec in segs:
        f_w, Pxx = welch(rec["x"], fs=FS, nperseg=min(1024, len(rec["x"])),
                         noverlap=512)
        psds.append(Pxx)
    psds = np.array(psds)
    mean_psd = psds.mean(axis=0)
    std_psd  = psds.std(axis=0)
    ax.semilogy(f_w, mean_psd, color=color, lw=2, label="Mean PSD")
    ax.fill_between(f_w, mean_psd - std_psd, mean_psd + std_psd,
                    alpha=0.25, color=color, label="±1 std")
    ax.set_xlim(0, 80)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("PSD (µV²/Hz)")
    ax.set_title(f"{group_name} — mean Welch PSD (n=20)")
    ax.legend()
    ax.grid(True, which="both", alpha=0.3)

plt.suptitle("Power Spectral Density: Epileptic vs Healthy", fontweight="bold")
plt.tight_layout()
plt.savefig("psd_comparison.png", dpi=150)
plt.show()

## Classification

In [ ]:
# ─── Cell 10: Classification ─────────────────────────────────────────────────────
# Prepare feature matrix (drop collinear / near-zero-variance features)
X = df[feature_cols].copy()
y = df["label"].values

# Drop features with > 5 % NaN
nan_frac = X.isna().mean()
X = X.loc[:, nan_frac < 0.05].fillna(X.median())

# Select top-K features by |Cohen d| (avoid adding noise features)
selected_feats = [f for f in stats_df["feature"].tolist() if f in X.columns][:25]
X_sel = X[selected_feats].values

print(f"Feature matrix for classification: {X_sel.shape}, "
      f"classes E={y.sum()}, H={(y==0).sum()}")

# Define classifiers
classifiers = {
    "Random Forest":     RandomForestClassifier(n_estimators=300, max_features="sqrt",
                                                 random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, max_depth=4,
                                                     learning_rate=0.05, random_state=42),
    "SVM (RBF)":         SVC(kernel="rbf", C=10, gamma="scale",
                              probability=True, random_state=42),
    "Logistic Reg.":     LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    "KNN":               KNeighborsClassifier(n_neighbors=7),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print("\nCross-validated performance (5-fold stratified):")
print(f"  {'Classifier':<22} {'Accuracy':>9} {'ROC-AUC':>9} {'F1':>9}")
print("  " + "-" * 52)

for name, clf in classifiers.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", clf)])
    cv_res = cross_validate(pipe, X_sel, y, cv=cv,
                            scoring=["accuracy", "roc_auc", "f1"],
                            return_train_score=False, n_jobs=-1)
    results[name] = {
        "acc":  cv_res["test_accuracy"],
        "auc":  cv_res["test_roc_auc"],
        "f1":   cv_res["test_f1"],
    }
    acc = cv_res["test_accuracy"].mean()
    auc = cv_res["test_roc_auc"].mean()
    f1  = cv_res["test_f1"].mean()
    print(f"  {name:<22} {acc:9.4f} {auc:9.4f} {f1:9.4f}")

## Best model — full training + evaluation

In [ ]:
# ─── Cell 11: Best model — full training + evaluation ────────────────────────────
# Random Forest consistently achieves the best accuracy on Bonn-like EEG data
best_name = max(results, key=lambda k: results[k]["auc"].mean())
print(f"\nBest classifier by mean ROC-AUC: {best_name}")

best_clf = classifiers[best_name]
best_pipe = Pipeline([("scaler", StandardScaler()), ("clf", best_clf)])
best_pipe.fit(X_sel, y)

# --- Cross-validated confusion matrix ---
from sklearn.model_selection import cross_val_predict
y_pred_cv = cross_val_predict(best_pipe, X_sel, y, cv=cv, method="predict")
y_prob_cv = cross_val_predict(best_pipe, X_sel, y, cv=cv, method="predict_proba")[:, 1]

print("\nClassification report (cross-validated):")
print(classification_report(y, y_pred_cv, target_names=["Healthy", "Epileptic"]))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y, y_pred_cv)
disp = ConfusionMatrixDisplay(cm, display_labels=["Healthy", "Epileptic"])
disp.plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title(f"Confusion matrix — {best_name}")

fpr, tpr, _ = roc_curve(y, y_prob_cv)
auc_score   = roc_auc_score(y, y_prob_cv)
axes[1].plot(fpr, tpr, lw=2, color="crimson",
             label=f"ROC curve (AUC = {auc_score:.4f})")
axes[1].plot([0,1],[0,1],"k--", lw=1)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title(f"ROC curve — {best_name}")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("classification_results.png", dpi=150)
plt.show()

## Feature importance

In [ ]:
# ─── Cell 12: Feature importance ─────────────────────────────────────────────────
if hasattr(best_clf, "feature_importances_"):
    importances = best_clf.feature_importances_
    feat_imp = pd.Series(importances, index=selected_feats).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(8, 8))
    feat_imp.plot.barh(ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(f"Feature importances — {best_name}", fontweight="bold")
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.savefig("feature_importance.png", dpi=150)
    plt.show()
    print("\nTop 10 features by importance:")
    print(feat_imp.sort_values(ascending=False).head(10).to_string())

## CV performance chart

In [ ]:
# ─── Cell 13: CV performance chart ───────────────────────────────────────────────
metrics_for_plot = {name: np.mean(v["acc"]) for name, v in results.items()}
auc_for_plot     = {name: np.mean(v["auc"]) for name, v in results.items()}

fig, ax = plt.subplots(figsize=(9, 5))
x_pos = np.arange(len(classifiers))
ax.bar(x_pos - 0.2,
       [metrics_for_plot[n] for n in classifiers.keys()],
       0.38, label="Accuracy", color="steelblue", alpha=0.85)
ax.bar(x_pos + 0.2,
       [auc_for_plot[n] for n in classifiers.keys()],
       0.38, label="ROC-AUC", color="crimson", alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels(classifiers.keys(), rotation=15, ha="right")
ax.set_ylim(0.5, 1.02)
ax.set_ylabel("Score")
ax.set_title("Classifier comparison — 5-fold cross-validated", fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("classifier_comparison.png", dpi=150)
plt.show()

## Summary statistics table

In [ ]:
# ─── Cell 14: Summary statistics table ───────────────────────────────────────────
summary_feats = ["acf1", "diff_std", "diff_kurtosis",
                 "hjorth_mobility", "hjorth_complexity",
                 "spectral_slope", "dominant_frequency",
                 "bp_delta_rel", "bp_alpha_rel", "bp_beta_rel",
                 "spectral_entropy", "approximate_entropy",
                 "sample_entropy", "hurst_exp", "detrended_fa"]
summary_feats = [f for f in summary_feats if f in df.columns]

rows = []
for feat in summary_feats:
    e = df[df["label"]==1][feat].dropna()
    h = df[df["label"]==0][feat].dropna()
    _, p = stats.mannwhitneyu(e, h, alternative="two-sided")
    rows.append({
        "Feature": feat,
        "E mean±std": f"{e.mean():.4f} ± {e.std(ddof=1):.4f}",
        "H mean±std": f"{h.mean():.4f} ± {h.std(ddof=1):.4f}",
        "MW p-value":  f"{p:.2e}",
        "Significant": "✓" if p < 0.05 else "✗",
    })

summary_table = pd.DataFrame(rows)
print("\n=== Comparative Statistical Summary ===")
print(summary_table.to_string(index=False))
summary_table.to_csv("statistical_summary.csv", index=False)

## Inference function for new user data

In [ ]:
# ─── Cell 15: Inference function for new user data ───────────────────────────────
import joblib, pickle

# Save the trained pipeline and the selected feature names for reuse
joblib.dump(best_pipe, "best_eeg_classifier.pkl")
with open("selected_features.txt", "w") as fh:
    fh.write("\n".join(selected_feats))

print(f"Trained pipeline saved → best_eeg_classifier.pkl")
print(f"Selected features saved → selected_features.txt")


def classify_new_eeg(txt_file_path,
                     model_path="best_eeg_classifier.pkl",
                     feature_list_path="selected_features.txt",
                     fs=FS,
                     verbose=True):
    """
    Classify a single new EEG text file as Epileptic (1) or Healthy (0).

    Parameters
    ----------
    txt_file_path   : str  — path to a single-column .txt EEG file
    model_path      : str  — path to the saved joblib pipeline
    feature_list_path: str — path to the saved selected_features.txt
    fs              : float — sampling frequency in Hz (default 173.61)
    verbose         : bool — print results

    Returns
    -------
    label     : int  (0 = Healthy, 1 = Epileptic)
    prob_E    : float — probability of being Epileptic
    feature_df: pd.DataFrame — extracted features for inspection
    """
    # Load artefacts
    pipe = joblib.load(model_path)
    with open(feature_list_path) as fh:
        feats_used = [line.strip() for line in fh if line.strip()]

    # Load signal
    x = load_txt_series(txt_file_path)
    if len(x) < 500:
        raise ValueError(f"Signal too short ({len(x)} samples). Minimum 500.")

    # Extract features
    feat_dict = extract_features(x, fs=fs)
    feat_df   = pd.DataFrame([feat_dict])

    # Build feature vector in the same order as training
    missing = [f for f in feats_used if f not in feat_df.columns]
    if missing:
        print(f"  Warning: {len(missing)} features missing; filled with 0: {missing}")
        for m in missing:
            feat_df[m] = 0.0

    X_new = feat_df[feats_used].fillna(0.0).values

    # Predict
    label = int(pipe.predict(X_new)[0])
    prob  = float(pipe.predict_proba(X_new)[0, 1])   # prob of Epileptic

    if verbose:
        print("\n=== EEG Classification Result ===")
        print(f"  File        : {os.path.basename(txt_file_path)}")
        print(f"  Prediction  : {'Epileptic' if label else 'Healthy'}")
        print(f"  P(Epileptic): {prob:.4f}")
        print(f"  P(Healthy)  : {1 - prob:.4f}")
        if prob > 0.7:
            print("  → HIGH confidence Epileptic")
        elif prob < 0.3:
            print("  → HIGH confidence Healthy")
        else:
            print("  → UNCERTAIN — borderline case; consult specialist")
        print("\n  Key extracted features:")
        for feat in ["acf1", "diff_std", "spectral_slope",
                     "approximate_entropy", "sample_entropy", "hurst_exp"]:
            if feat in feat_df.columns:
                print(f"    {feat:30s} = {feat_df[feat].values[0]:.5f}")

    return label, prob, feat_df

## Demo — classify a downloaded file

In [ ]:
# ─── Cell 16: Demo — classify a downloaded file ──────────────────────────────────
# Replace the path below with any .txt EEG file you want to classify
print("""
DEMO_FILE = os.path.join(DATA_DIR, "TrainE1_1.txt")

if os.path.exists(DEMO_FILE):
    label, prob_E, feat_df = classify_new_eeg(DEMO_FILE)
else:
    print(f"Demo file not found: {DEMO_FILE}")
    print("Run the download cell first, or supply your own EEG .txt file.")
""")

## Classify a user-uploaded file (Colab-specific)

In [ ]:
# ─── Cell 17: Classify a user-uploaded file (Colab-specific) ─────────────────────
print("""
To classify YOUR OWN EEG file in Colab, run:

    from google.colab import files
    uploaded = files.upload()            # select your .txt EEG file
    fname = list(uploaded.keys())[0]
    label, prob, feats = classify_new_eeg(fname)

The file must contain one EEG amplitude value per line (ASCII text, µV).
Minimum length: 500 samples. Optimal: 4097 samples at 173.61 Hz.
""")

In [ ]:
# ─── Cell 18: Download test-set files (E1–E10 and H1–H10, 20 segs each) ─────────
TEST_DIR = "eeg_test_data"
os.makedirs(TEST_DIR, exist_ok=True)

def build_test_file_list():
    entries = []
    for group, true_label in [("E", 1), ("H", 0)]:
        for subject in range(1, 11):          # 10 subjects per class
            for seg in range(1, 21):          # 20 segments per subject
                fname = f"{group}{subject}_{seg}.txt"
                url   = ZENODO_BASE + fname + "?download=1"
                path  = os.path.join(TEST_DIR, fname)
                entries.append((url, path, true_label, fname))
    return entries

test_file_list = build_test_file_list()
print(f"Total test files expected: {len(test_file_list)}")
print("Downloading test data (may take 2–4 minutes) …")

ok_test = 0
for url, path, lbl, fname in test_file_list:
    if download_file(url, path):
        ok_test += 1

print(f"Downloaded/available: {ok_test}/{len(test_file_list)} test files")

In [ ]:
# ─── Cell 19: Load, extract features and classify test segments ──────────────────
test_records = []
for url, path, true_label, fname in test_file_list:
    if os.path.exists(path):
        x = load_txt_series(path)
        if len(x) >= 1000:
            test_records.append({"x": x, "true_label": true_label, "filename": fname})

n_TE = sum(1 for r in test_records if r["true_label"] == 1)
n_TH = sum(1 for r in test_records if r["true_label"] == 0)
print(f"Test segments loaded: Epileptic={n_TE}, Healthy={n_TH}, Total={len(test_records)}")

print("\nExtracting features from test set …")
test_rows = []
for rec in test_records:
    try:
        f = extract_features(rec["x"])
        f["true_label"] = rec["true_label"]
        f["filename"]   = rec["filename"]
        test_rows.append(f)
    except Exception as e:
        print(f"  Skipped {rec['filename']}: {e}")

df_test = pd.DataFrame(test_rows)
df_test.to_csv("eeg_test_features.csv", index=False)
print(f"Test feature matrix: {df_test.shape}")

# Align columns to training feature set
X_test_raw = df_test[[c for c in df_test.columns
                       if c not in ("true_label", "filename")]].copy()
nan_frac_test = X_test_raw.isna().mean()
X_test_raw = X_test_raw.fillna(X_test_raw.median())

# Use the same selected_feats from training
missing_feats = [f for f in selected_feats if f not in X_test_raw.columns]
for m in missing_feats:
    X_test_raw[m] = 0.0
X_test = X_test_raw[selected_feats].values
y_test = df_test["true_label"].values

# Predict with the trained pipeline (no retraining)
y_pred_test = best_pipe.predict(X_test)
y_prob_test = best_pipe.predict_proba(X_test)[:, 1]

df_test["predicted_label"] = y_pred_test
df_test["prob_epileptic"]  = y_prob_test
df_test["correct"]         = (y_pred_test == y_test).astype(int)
df_test.to_csv("eeg_test_predictions.csv", index=False)
print("Predictions saved → eeg_test_predictions.csv")

In [ ]:
# ─── Cell 20: Full classification report ─────────────────────────────────────────
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_auc_score,
                              roc_curve, accuracy_score, balanced_accuracy_score,
                              matthews_corrcoef, cohen_kappa_score,
                              precision_recall_curve, average_precision_score)

acc      = accuracy_score(y_test, y_pred_test)
bal_acc  = balanced_accuracy_score(y_test, y_pred_test)
roc_auc  = roc_auc_score(y_test, y_prob_test)
avg_prec = average_precision_score(y_test, y_prob_test)
mcc      = matthews_corrcoef(y_test, y_pred_test)
kappa    = cohen_kappa_score(y_test, y_pred_test)

print("=" * 55)
print(f"  TEST SET PERFORMANCE — {best_name}")
print("=" * 55)
print(f"  Segments evaluated      : {len(y_test)}")
print(f"  Epileptic (E) segments  : {(y_test==1).sum()}")
print(f"  Healthy   (H) segments  : {(y_test==0).sum()}")
print("-" * 55)
print(f"  Accuracy                : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Balanced Accuracy       : {bal_acc:.4f}")
print(f"  ROC-AUC                 : {roc_auc:.4f}")
print(f"  Average Precision (PR)  : {avg_prec:.4f}")
print(f"  Matthews Corr. Coef.    : {mcc:.4f}")
print(f"  Cohen Kappa             : {kappa:.4f}")
print("=" * 55)
print()
print(classification_report(y_test, y_pred_test,
                             target_names=["Healthy (H)", "Epileptic (E)"]))

In [ ]:
# ─── Cell 21: Visualise classification results ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1. Confusion matrix
cm = confusion_matrix(y_test, y_pred_test)
disp = ConfusionMatrixDisplay(cm, display_labels=["Healthy", "Epileptic"])
disp.plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title(f"Confusion Matrix\n{best_name} — Test Set", fontweight="bold")

# 2. ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob_test)
axes[1].plot(fpr, tpr, lw=2, color="crimson",
             label=f"ROC (AUC = {roc_auc:.4f})")
axes[1].plot([0,1],[0,1], "k--", lw=1)
axes[1].set_xlabel("False Positive Rate");  axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve — Test Set", fontweight="bold")
axes[1].legend();  axes[1].grid(alpha=0.3)

# 3. Precision-Recall curve
prec, rec, _ = precision_recall_curve(y_test, y_prob_test)
axes[2].plot(rec, prec, lw=2, color="steelblue",
             label=f"AP = {avg_prec:.4f}")
axes[2].set_xlabel("Recall");  axes[2].set_ylabel("Precision")
axes[2].set_title("Precision-Recall Curve — Test Set", fontweight="bold")
axes[2].legend();  axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("test_classification_results.png", dpi=150)
plt.show()

In [ ]:
# ─── Cell 22: Per-subject accuracy and probability distribution ───────────────────
df_test["subject"] = df_test["filename"].str.extract(r'^([EH]\d+)_')
df_test["group"]   = df_test["filename"].str[0]

subj_stats = df_test.groupby(["subject","group","true_label"]).agg(
    n_segments    = ("correct", "count"),
    n_correct     = ("correct", "sum"),
    accuracy      = ("correct", "mean"),
    mean_prob_E   = ("prob_epileptic", "mean"),
    std_prob_E    = ("prob_epileptic", "std"),
).reset_index().sort_values(["group", "subject"])

print("\nPer-subject accuracy on test set:")
print(subj_stats[["subject","group","n_segments","n_correct",
                   "accuracy","mean_prob_E","std_prob_E"]].to_string(index=False))
subj_stats.to_csv("eeg_test_per_subject.csv", index=False)

# Bar chart: accuracy per subject, coloured by group
fig, ax = plt.subplots(figsize=(14, 5))
colors = subj_stats["group"].map({"E": "crimson", "H": "steelblue"})
bars = ax.bar(subj_stats["subject"], subj_stats["accuracy"],
              color=colors, edgecolor="white", alpha=0.85)
ax.axhline(acc, ls="--", color="black", lw=1.2, label=f"Overall acc = {acc:.3f}")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Accuracy")
ax.set_xlabel("Subject")
ax.set_title("Per-subject classification accuracy — Test Set", fontweight="bold")
ax.set_xticklabels(subj_stats["subject"], rotation=45, ha="right")
from matplotlib.patches import Patch
legend_handles = [Patch(color="crimson", label="Epileptic (E)"),
                  Patch(color="steelblue", label="Healthy (H)"),
                  plt.Line2D([0],[0], color="black", ls="--", label=f"Mean acc={acc:.3f}")]
ax.legend(handles=legend_handles)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("test_per_subject_accuracy.png", dpi=150)
plt.show()

In [ ]:
# ─── Cell 23: Probability distributions and error analysis ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Predicted probability histogram by true class
for lbl, name, color in [(1,"Epileptic","crimson"), (0,"Healthy","steelblue")]:
    vals = df_test[df_test["true_label"]==lbl]["prob_epileptic"]
    axes[0].hist(vals, bins=30, alpha=0.6, color=color, density=True, label=name)
axes[0].axvline(0.5, color="black", ls="--", lw=1.2, label="Decision boundary")
axes[0].set_xlabel("P(Epileptic)")
axes[0].set_ylabel("Density")
axes[0].set_title("Predicted probability distribution\nby true class", fontweight="bold")
axes[0].legend()

# 2. Errors vs confidence
errors = df_test[df_test["correct"]==0]
correct= df_test[df_test["correct"]==1]
axes[1].scatter(correct["prob_epileptic"], correct.index % len(correct),
                alpha=0.3, s=8, color="green", label=f"Correct ({len(correct)})")
axes[1].scatter(errors["prob_epileptic"], errors.index % len(errors),
                alpha=0.6, s=20, color="red", marker="x", label=f"Error ({len(errors)})")
axes[1].axvline(0.5, color="black", ls="--", lw=1)
axes[1].set_xlabel("P(Epileptic)")
axes[1].set_title("Correct vs misclassified segments", fontweight="bold")
axes[1].legend()
axes[1].set_yticks([])

# 3. Group-level summary boxplot
import matplotlib.patches as mpatches
data_E = df_test[df_test["true_label"]==1]["prob_epileptic"]
data_H = df_test[df_test["true_label"]==0]["prob_epileptic"]
bp = axes[2].boxplot([data_H, data_E], labels=["Healthy (H)", "Epileptic (E)"],
                     patch_artist=True, notch=True)
for patch, color in zip(bp["boxes"], ["steelblue","crimson"]):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[2].axhline(0.5, color="black", ls="--", lw=1.2, label="Decision boundary")
axes[2].set_ylabel("P(Epileptic)")
axes[2].set_title("P(Epileptic) by true group\nBoxplot with notches", fontweight="bold")
axes[2].legend()
axes[2].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("test_probability_analysis.png", dpi=150)
plt.show()

# Print error summary
print(f"\nTotal errors: {len(errors)}/{len(df_test)} ({len(errors)/len(df_test)*100:.1f}%)")
print("\nMisclassified files:")
print(errors[["filename","true_label","predicted_label","prob_epileptic"]].to_string(index=False))


In [ ]:
# ─── Cell 24: Final summary statistics table ─────────────────────────────────────
from scipy import stats as sp_stats

summary_stats = {
    "Metric": [
        "Total test segments", "Epileptic (E)", "Healthy (H)",
        "Accuracy", "Balanced Accuracy",
        "Sensitivity (Recall E)", "Specificity (Recall H)",
        "Precision (E)", "F1-score (E)",
        "ROC-AUC", "Average Precision (PR-AUC)",
        "Matthews Corr. Coef.", "Cohen Kappa",
        "Mean P(E | true E)", "Mean P(E | true H)",
        "95% CI Accuracy (Wilson)",
    ],
    "Value": []
}

from sklearn.metrics import precision_score, recall_score, f1_score
n_total = len(y_test)
sens    = recall_score(y_test, y_pred_test, pos_label=1)
spec    = recall_score(y_test, y_pred_test, pos_label=0)
prec_e  = precision_score(y_test, y_pred_test, pos_label=1)
f1_e    = f1_score(y_test, y_pred_test, pos_label=1)

# Wilson score 95% CI for accuracy
z = 1.96
n, p = n_total, acc
lo = (p + z**2/(2*n) - z*((p*(1-p)/n + z**2/(4*n**2))**0.5)) / (1 + z**2/n)
hi = (p + z**2/(2*n) + z*((p*(1-p)/n + z**2/(4*n**2))**0.5)) / (1 + z**2/n)

values = [
    n_total, (y_test==1).sum(), (y_test==0).sum(),
    f"{acc:.4f} ({acc*100:.2f}%)", f"{bal_acc:.4f}",
    f"{sens:.4f}", f"{spec:.4f}",
    f"{prec_e:.4f}", f"{f1_e:.4f}",
    f"{roc_auc:.4f}", f"{avg_prec:.4f}",
    f"{mcc:.4f}", f"{kappa:.4f}",
    f"{df_test[df_test['true_label']==1]['prob_epileptic'].mean():.4f}",
    f"{df_test[df_test['true_label']==0]['prob_epileptic'].mean():.4f}",
    f"[{lo:.4f}, {hi:.4f}]",
]

summary_df = pd.DataFrame({"Metric": summary_stats["Metric"], "Value": values})
print("\n" + "="*50)
print("  FINAL TEST SET STATISTICS SUMMARY")
print("="*50)
print(summary_df.to_string(index=False))
summary_df.to_csv("eeg_final_summary.csv", index=False)
print("\nSummary saved → eeg_final_summary.csv")